# Extending Elastic Agent Builder with LlamaExtract

We combine **LlamaExtract** (schema-driven PDF extraction) with **Elastic Agent Builder**
(workflow tools + search) to process complex documents like the
[World Bank Global Economic Prospects (Jan 2026)](https://www.worldbank.org/en/publication/global-economic-prospects).

Pipeline: Define schema → Create extraction agent → Build workflow → Create agent → Ask questions about the data.

## 1. Setup

In [ ]:
%pip install llama-cloud elasticsearch dotenv pydantic


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Set your API keys and URLs in a `.env` file:
```bash
LLAMA_CLOUD_API_KEY="llx-..."
ELASTICSEARCH_URL="https://..."
ELASTICSEARCH_API_KEY="your-elastic-api-key"
KIBANA_URL="https://your-kibana-url"
```

In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
LLAMA_CLOUD_API_KEY = os.getenv("LLAMA_CLOUD_API_KEY")
KIBANA_URL = os.getenv("KIBANA_URL")

## 2. Define the Extraction Schema

Pydantic model that defines what fields to extract from the PDF.

In [4]:
from pydantic import BaseModel, Field


class EconomicIndicator(BaseModel):
    indicator: str = Field(
        description="Name of the economic indicator (e.g., 'Annual growth of per capita investment')"
    )
    emerging_markets_value: str = Field(description="Value for emerging markets (EM)")
    frontier_markets_value: str = Field(description="Value for frontier markets (FM)")
    other_developing_value: str = Field(
        description="Value for other developing economies (ODE), if available"
    )
    period: str = Field(description="Time period for this data point (e.g., '2020-24')")


class CountryExample(BaseModel):
    country: str = Field(description="Country name")
    development_approach: str = Field(
        description="The development strategy or sector focus of this country"
    )


class GlobalEconomicReport(BaseModel):
    report_title: str = Field(description="Full title of the report")
    publication_date: str = Field(description="Publication date (e.g., 'January 2026')")
    publisher: str = Field(description="Publishing organization")
    main_topic: str = Field(
        description="The primary topic or focus of the executive summary"
    )
    executive_summary: str = Field(
        description="A concise summary of the executive summary's main findings and conclusions"
    )
    frontier_markets_population_share_2025: str = Field(
        description="Frontier markets' share of global population in 2025"
    )
    frontier_markets_gdp_share_2025: str = Field(
        description="Frontier markets' share of global GDP in 2025"
    )
    frontier_markets_gdp_per_capita_vs_emerging: str = Field(
        description="How frontier markets' GDP per capita compares to emerging markets"
    )
    per_capita_investment_growth_2000s: str = Field(
        description="Average annual per capita investment growth in frontier markets during the 2000s"
    )
    per_capita_investment_growth_2020s: str = Field(
        description="Average annual per capita investment growth in frontier markets in the early 2020s"
    )
    poverty_rate_comparison: str = Field(
        description="How poverty rates in frontier markets compare to emerging markets"
    )
    sovereign_defaults_trend: str = Field(
        description="Description of the trend in sovereign default events for frontier markets"
    )
    key_vulnerabilities: list[str] = Field(
        description="List of key vulnerabilities and risks facing frontier markets"
    )
    policy_recommendations: list[str] = Field(
        description="List of policy recommendations for frontier market policy makers"
    )
    top_performing_countries: list[CountryExample] = Field(
        description="Examples of top-performing frontier markets and their development approaches"
    )
    countries_graduated_to_high_income: list[str] = Field(
        description="Countries that graduated from frontier market to emerging market status by reaching high-income level"
    )
    economic_indicators: list[EconomicIndicator] = Field(
        description="Key economic indicators extracted from charts and tables (Figures ES.A-F)"
    )


print(GlobalEconomicReport.model_json_schema())

{'$defs': {'CountryExample': {'properties': {'country': {'description': 'Country name', 'title': 'Country', 'type': 'string'}, 'development_approach': {'description': 'The development strategy or sector focus of this country', 'title': 'Development Approach', 'type': 'string'}}, 'required': ['country', 'development_approach'], 'title': 'CountryExample', 'type': 'object'}, 'EconomicIndicator': {'properties': {'indicator': {'description': "Name of the economic indicator (e.g., 'Annual growth of per capita investment')", 'title': 'Indicator', 'type': 'string'}, 'emerging_markets_value': {'description': 'Value for emerging markets (EM)', 'title': 'Emerging Markets Value', 'type': 'string'}, 'frontier_markets_value': {'description': 'Value for frontier markets (FM)', 'title': 'Frontier Markets Value', 'type': 'string'}, 'other_developing_value': {'description': 'Value for other developing economies (ODE), if available', 'title': 'Other Developing Value', 'type': 'string'}, 'period': {'descr

## 3. Create the Extraction Agent

Create a LlamaExtract agent configured with our schema. Save the `agent.id` — you'll need it for the workflow.

In [7]:
from llama_cloud import LlamaCloud

client = LlamaCloud(api_key=LLAMA_CLOUD_API_KEY)

# Create the extraction agent with our schema
agent = client.extraction.extraction_agents.create(
    name="global-economic-prospects-extractor",
    data_schema=GlobalEconomicReport.model_json_schema(),
    config={},
)
print(f"Created extraction agent: {agent.id}")

Created extraction agent: e1f9fa9a-7bf0-4a2b-8d4e-208b0ebf7674


## 4. Create the Elasticsearch Index

Create the index with mappings matching our extraction schema. The workflow will index documents here.

In [19]:
from elasticsearch import Elasticsearch

es = Elasticsearch(
    ELASTICSEARCH_URL,
    api_key=ELASTICSEARCH_API_KEY,
)

print(f"Connected: {es.info()['cluster_name']}")

Connected: fd31ad5f77d841d69b622c17ed64b766


In [ ]:
INDEX_NAME = "economic-reports"

mappings = {
    "mappings": {
        "properties": {
            "report_title": {"type": "text"},
            "publication_date": {"type": "keyword"},
            "publisher": {"type": "keyword"},
            "main_topic": {"type": "text"},
            "executive_summary": {"type": "text"},
            "frontier_markets_population_share_2025": {"type": "keyword"},
            "frontier_markets_gdp_share_2025": {"type": "keyword"},
            "frontier_markets_gdp_per_capita_vs_emerging": {"type": "text"},
            "per_capita_investment_growth_2000s": {"type": "keyword"},
            "per_capita_investment_growth_2020s": {"type": "keyword"},
            "poverty_rate_comparison": {"type": "text"},
            "sovereign_defaults_trend": {"type": "text"},
            "key_vulnerabilities": {"type": "text"},
            "policy_recommendations": {"type": "text"},
            "top_performing_countries": {
                "type": "nested",
                "properties": {
                    "country": {"type": "keyword"},
                    "development_approach": {"type": "text"},
                },
            },
            "countries_graduated_to_high_income": {"type": "keyword"},
            "economic_indicators": {
                "type": "nested",
                "properties": {
                    "indicator": {"type": "text"},
                    "emerging_markets_value": {"type": "keyword"},
                    "frontier_markets_value": {"type": "keyword"},
                    "other_developing_value": {"type": "keyword"},
                    "period": {"type": "keyword"},
                },
            },
            "source_url": {"type": "keyword"},
        }
    }
}

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

es.indices.create(index=INDEX_NAME, body=mappings)
print(f"Created index: {INDEX_NAME}")

## 5. Workflow Tool for Elastic Agent Builder

This workflow automates the full pipeline: receive a PDF URL → extract with LlamaExtract → index into Elasticsearch.

**Manual setup required:** Copy the YAML below and paste it in the Elastic UI at
**Management → AI Assistants & Agents → Workflows → Create Workflow**. You'll need to
manually insert:
- `pdf_url`: a public URL pointing directly to the PDF (e.g. a `raw.githubusercontent.com` link)
- `extraction_agent_id`: the agent ID from step 3 above
- `document_id`: a unique ID you choose for the indexed document
- `llama_cloud_api_key`: add it under **Secrets** in the workflow config

```yaml
version: "1"
name: LlamaExtract Economic Report Processor
description: >
  Receives a PDF URL, extracts structured economic data using the
  LlamaExtract API, and indexes the results into Elasticsearch.
enabled: true

inputs:
  - name: pdf_url
    type: string
    description: Public URL of the PDF document to process
    required: true
  - name: extraction_agent_id
    type: string
    description: LlamaExtract extraction agent ID (pre-configured with schema)
    required: true
  - name: document_id
    type: string
    description: Unique identifier for the indexed document
    required: true

consts:
  indexName: economic-reports
  llamaExtractBaseUrl: https://api.cloud.llamaindex.ai/api/v1

triggers:
  - type: manual

steps:
  - name: upload_pdf
    type: http
    with:
      url: "{{ consts.llamaExtractBaseUrl }}/files"
      method: POST
      headers:
        Authorization: "Bearer {{ secrets.llama_cloud_api_key }}"
        Content-Type: application/json
      body: |
        {
          "url": "{{ inputs.pdf_url }}",
          "purpose": "extract"
        }
      timeout: 60s

  - name: create_extraction_job
    type: http
    with:
      url: "{{ consts.llamaExtractBaseUrl }}/extraction/jobs"
      method: POST
      headers:
        Authorization: "Bearer {{ secrets.llama_cloud_api_key }}"
        Content-Type: application/json
      body: |
        {
          "extraction_agent_id": "{{ inputs.extraction_agent_id }}",
          "file_id": "{{ steps.upload_pdf.output.data.id }}"
        }
      timeout: 30s

  - name: get_extraction_result
    type: http
    with:
      url: "{{ consts.llamaExtractBaseUrl }}/extraction/jobs/{{ steps.create_extraction_job.output.data.id }}/result"
      method: GET
      headers:
        Authorization: "Bearer {{ secrets.llama_cloud_api_key }}"
        Accept: application/json
      timeout: 120s

  - name: index_extracted_data
    type: elasticsearch.index
    with:
      index: "{{ consts.indexName }}"
      id: "{{ inputs.document_id }}"
      document:
        report_title: "{{ steps.get_extraction_result.output.data.report_title }}"
        publication_date: "{{ steps.get_extraction_result.output.data.publication_date }}"
        publisher: "{{ steps.get_extraction_result.output.data.publisher }}"
        main_topic: "{{ steps.get_extraction_result.output.data.main_topic }}"
        executive_summary: "{{ steps.get_extraction_result.output.data.executive_summary }}"
        frontier_markets_population_share_2025: "{{ steps.get_extraction_result.output.data.frontier_markets_population_share_2025 }}"
        frontier_markets_gdp_share_2025: "{{ steps.get_extraction_result.output.data.frontier_markets_gdp_share_2025 }}"
        frontier_markets_gdp_per_capita_vs_emerging: "{{ steps.get_extraction_result.output.data.frontier_markets_gdp_per_capita_vs_emerging }}"
        per_capita_investment_growth_2000s: "{{ steps.get_extraction_result.output.data.per_capita_investment_growth_2000s }}"
        per_capita_investment_growth_2020s: "{{ steps.get_extraction_result.output.data.per_capita_investment_growth_2020s }}"
        poverty_rate_comparison: "{{ steps.get_extraction_result.output.data.poverty_rate_comparison }}"
        sovereign_defaults_trend: "{{ steps.get_extraction_result.output.data.sovereign_defaults_trend }}"
        key_vulnerabilities: "{{ steps.get_extraction_result.output.data.key_vulnerabilities | json }}"
        policy_recommendations: "{{ steps.get_extraction_result.output.data.policy_recommendations | json }}"
        top_performing_countries: "{{ steps.get_extraction_result.output.data.top_performing_countries | json }}"
        countries_graduated_to_high_income: "{{ steps.get_extraction_result.output.data.countries_graduated_to_high_income | json }}"
        economic_indicators: "{{ steps.get_extraction_result.output.data.economic_indicators | json }}"
        source_url: "{{ inputs.pdf_url }}"
      refresh: wait_for

  - name: verify_document
    type: elasticsearch.search
    with:
      index: "{{ consts.indexName }}"
      query:
        term:
          _id: "{{ inputs.document_id }}"
```

## 6. Create the Agent via API

Create an ES|QL tool to query the indexed data, then create an agent that uses it.
We use the [Agent Builder Kibana APIs](https://www.elastic.co/docs/explore-analyze/ai-features/agent-builder/kibana-api).

In [ ]:
import requests

headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

In [ ]:
# Create an ES|QL tool to query economic reports
tool_payload = {
    "id": "query_economic_reports",
    "type": "esql",
    "description": "Search economic report data extracted from PDFs. Use this to answer questions about frontier markets, economic indicators, policy recommendations, and country performance.",
    "configuration": {
        "query": "FROM economic-reports | WHERE MATCH(executive_summary, ?query) OR MATCH(key_vulnerabilities, ?query) OR MATCH(policy_recommendations, ?query) | LIMIT 5",
        "params": {
            "query": {
                "type": "keyword",
                "description": "The search query to find relevant economic data"
            }
        }
    },
    "tags": ["economic-data", "llama-extract"]
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=tool_payload,
)
print(f"Tool created: {response.status_code}")
print(response.json())

In [ ]:
# Create the agent with the tool
agent_payload = {
    "id": "economic-report-analyst",
    "name": "Economic Report Analyst",
    "description": "Analyzes economic reports extracted from PDFs and answers questions about the data.",
    "labels": ["economics", "llama-extract"],
    "configuration": {
        "instructions": (
            "You are an economic research assistant. Use the query_economic_reports tool "
            "to search extracted report data and answer questions accurately. "
            "When presenting data, use clear formatting with bullet points or tables. "
            "Always cite the report title and publication date in your answers."
        ),
        "tools": [
            {
                "tool_ids": [
                    "query_economic_reports",
                    "platform.core.search",
                ]
            }
        ]
    }
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/agents",
    headers=headers,
    json=agent_payload,
)
print(f"Agent created: {response.status_code}")
print(response.json())

In [ ]:
# Ask the agent a question about the extracted data
converse_payload = {
    "agent_id": "economic-report-analyst",
    "input": "What are the main vulnerabilities facing frontier markets according to the World Bank report?",
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/converse",
    headers=headers,
    json=converse_payload,
)
answer = response.json()
print(answer)

## 7. Verify Indexed Data

After the agent processes a PDF via the workflow, verify the extracted data was indexed correctly.

In [ ]:
# Overview of indexed reports
result = es.search(index=INDEX_NAME, query={"match_all": {}})

for hit in result["hits"]["hits"]:
    report = hit["_source"]
    print(f"Report: {report['report_title']}")
    print(f"Publisher: {report['publisher']}")
    print(f"Topic: {report['main_topic']}")
    print(f"FM Population Share: {report.get('frontier_markets_population_share_2025')}")
    print(f"FM GDP Share: {report.get('frontier_markets_gdp_share_2025')}")

In [ ]:
# Query: Search for vulnerability-related content
result = es.search(
    index=INDEX_NAME,
    query={"match": {"key_vulnerabilities": "sovereign default fiscal"}},
)

for hit in result["hits"]["hits"]:
    print(f"Report: {hit['_source']['report_title']}")
    print(f"Sovereign defaults trend: {hit['_source']['sovereign_defaults_trend']}")
    print("\nKey vulnerabilities:")
    for vuln in hit["_source"]["key_vulnerabilities"]:
        print(f"  - {vuln}")

In [ ]:
# Query: Find top-performing countries using nested query
result = es.search(
    index=INDEX_NAME,
    query={
        "nested": {
            "path": "top_performing_countries",
            "query": {
                "match": {
                    "top_performing_countries.development_approach": "manufacturing"
                }
            },
        }
    },
)

if result["hits"]["hits"]:
    countries = result["hits"]["hits"][0]["_source"]["top_performing_countries"]
    print("Top-performing frontier markets:")
    for country in countries:
        print(f"  {country['country']}: {country['development_approach']}")

In [ ]:
# Query: Get economic indicators from charts
result = es.search(
    index=INDEX_NAME,
    query={
        "nested": {
            "path": "economic_indicators",
            "query": {"match": {"economic_indicators.indicator": "investment growth"}},
        }
    },
)

if result["hits"]["hits"]:
    indicators = result["hits"]["hits"][0]["_source"]["economic_indicators"]
    print("Economic indicators extracted from charts:")
    for ind in indicators:
        print(f"  {ind['indicator']} ({ind['period']}):")
        print(
            f"    EM: {ind['emerging_markets_value']}, FM: {ind['frontier_markets_value']}"
        )

## Summary

This notebook sets up a pipeline where **LlamaExtract** handles schema-driven PDF extraction
and **Elastic Agent Builder** automates the indexing via a workflow tool. Once configured,
an agent can process new reports on demand and query the structured data in Elasticsearch.